# 歌词分词，词性标注

In [1]:
import json
import pandas as pd


# import thulac


from collections import Counter
from openai import OpenAI

In [2]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [3]:
import sys
sys.path.append('..')

# 分词，词频与词性分析

In [4]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v',
    '春娇': 'n',
    '学会': 'v'
}

In [5]:
# def process_lyrics_with_jieba(text):
#     # 1. 词性标注与分词
#     # jieba.posseg 会同时返回词和词性
#     words_with_pos = pseg.cut(text)

    
#     # 2. 过滤无意义字符（标点、空格、单字符停用词）
#     filtered_data = []
#     for word, pos in words_with_pos:
#         # 排除标点符号（x表示标点）及空白字符
#         if pos != 'x' and len(word.strip()) > 0:
#             if word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 3. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 4. 汇总信息 (词, 词性, 频数)
#     # 我们以词为 Key，存储词性
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     # 排序：按词频从高到低
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count # 词频
#         })
    
#     return sorted_results

In [6]:
import re
from hanlp_restful import HanLPClient
HanLP = HanLPClient('https://www.hanlp.com/hanlp/v21/redirect', auth="699691e7eaf61a3aca90d7b8", language='zh')

def is_chinese_word(word):
    """
    判断是否为纯中文词
    """
    return 1 if re.fullmatch(r'[\u4e00-\u9fff]+', word) else 0


def process_lyrics_with_hanlp_multi_pos(text, word_to_fix=None):
    if not text:
        return []
    
    # 调用 HanLP
    result = HanLP.parse(text, tasks='pos/pku')
    
    sentences = result['tok/fine']
    pos_sentences = result['pos/pku']
    
    # 统计 (word, pos) -> freq
    word_pos_counter = Counter()
    
    for words, pos_tags in zip(sentences, pos_sentences):
        for word, tag in zip(words, pos_tags):
            
            word = word.strip()
            
            # 过滤标点
            if tag == 'w' or not word:
                continue
            
            # 词性修正
            if word_to_fix and word in word_to_fix:
                tag = word_to_fix[word]
            
            word_pos_counter[(word, tag)] += 1
    
    # 构建结果列表
    results = []
    for (word, pos), freq in word_pos_counter.items():
        results.append({
            "word": word,
            "pos": pos,
            "freq": freq,
            "is_chinese": is_chinese_word(word)
        })
    
    # 按词频排序
    results.sort(key=lambda x: x["freq"], reverse=True)
    
    return results


In [7]:
# thu = thulac.thulac(seg_only=False, filt=True) 

# def process_lyrics_with_thulac(text, word_to_fix=None):
#     if not text:
#         return []
    
#     # 2. 执行分词与词性标注
#     # 返回格式为 [[word, pos], [word, pos], ...]
#     words_with_pos = thu.cut(text)
    
#     # 3. 过滤无意义字符与词性修正
#     # thulac 的标点词性通常是 'w'
#     filtered_data = []
#     for word, pos in words_with_pos:
#         word = word.strip()
#         # 排除标点符号、空白字符
#         if pos != 'w' and len(word) > 0:
#             # 逻辑修正：word_to_fix 通常是修正词性
#             if word_to_fix and word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 4. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 5. 汇总信息
#     # 建立 word -> pos 映射
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count
#         })
    
#     return sorted_results

In [8]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'
    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            # lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
            #     i['lyrics_text'], word_to_fix=word_to_fix)
            print(i['song_name'])
            lyric_words_dict[i['song_id']] = process_lyrics_with_hanlp_multi_pos(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    return df_word

In [9]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word = df_word.copy()
    df_songs = df_songs.copy()
    df_word['song_id'] = df_word['song_id'].astype(str)
    df_word['word'] = df_word['word'].astype(str)
    df_word['is_chinese'] = df_word['word'].apply(is_chinese_word)

    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    df_merged = df_word.merge(df_songs, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# 批量采集

In [69]:
singers = [('luodayou', '罗大佑'), ('lizongsheng', '李宗盛'), ('zhangxueyou', '张学友'), ('twins', 'Twins'), ('wangsulong', '汪苏泷'), ('panweibo', '潘玮柏'), ('dengziqi', 'G.E.M. 邓紫棋'), ('xuezhiqian', '薛之谦'), ('xusong', '许嵩'), ('zhangjie', '张杰'), ('taozhe', '陶喆'), ('fangdatong', '方大同'), ('wangfei', '王菲'), ('maobuyi', '毛不易'), ('beyond', 'BEYOND')]
for i in singers[-1:]:
    file_path_prefix = f'data/{i[0]}/'
    # 歌曲数据
    df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
    # 词性解析
    # hanlp暂时不需要word_to_fix
    df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
    df_word.to_csv(file_path_prefix + "raw_words_data.csv", index=False)
    # 重新读取
    df_word_read = pd.read_csv(file_path_prefix + "raw_words_data.csv")
    df_merged = words_data_merge(df_word_read, df_songs)
    df_merged = df_merged.dropna(subset='song_name')
    # 过滤中文词
    df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
    # 虚拟专辑数据
    df_songs_part = df_merged_chn[[
        'song_name_pure'
    ]].drop_duplicates(keep='first').reset_index(drop=True)
    # 只保留120个
    df_songs_part = df_songs_part.head(100)
    df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                            1).astype(str)
    df_songs_part['album_order'] = df_songs_part.index // 10
    # 虚拟专辑数据，index//12+1作为虚拟专辑
    df_merged_chn = df_merged_chn.copy()
    df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
    df_merged_chn = df_merged_chn.drop(columns=['album_name'])
    df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
    # 删除album_order为空的数据
    df_merged_chn = df_merged_chn.dropna(subset=['album_order'], axis=0)
    df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)
    df_songs_final = df_merged_chn.drop(columns=['word', 'pos', 'freq', 'is_chinese']).drop_duplicates().reset_index(drop=True)
    df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

海阔天空
Amani (粤语)
灰色轨迹
光辉岁月
真的爱你
情人
不再犹豫
午夜怨曲
喜欢你
冷雨夜
大地 (粤语)
农民
灰色軌跡 (Grey Trail)
无尽空虚
我是愤怒
无悔这一生
遥望
逝去日子
再见理想
曾是拥有
无泪的遗撼
旧日的足迹
早班火车
报答一生
冲开一切
俾面派对
不可一世
无泪的遗憾
无语问苍天
为了你 为了我
可否冲破
和平与爱
交织千个心
谁伴我闯荡
半斤八两
愿我能
未曾后悔
遥かなる夢に
原谅我今天
遥远的Paradise
无声的告别
命运是你家
岁月无声 (国语)
赤红热血
长空
高温派对
喜欢妳
为了你，为了我
全是爱
真的爱妳
是错也再不分
走不开的快乐
曾经拥有
十八
昔日舞曲
孤单一吻
爆裂都市
点解 点解
想你
亚拉伯跳舞女郎
爸爸妈妈
总有爱
活着便精彩
完全的拥有
天真的创伤
文武英杰宣言
战胜心魔
荒谬
午夜迷墙
冲上云霄
遥かなる梦に〜Far away〜
秘密警察
雾
脑部侵袭
大厦
又是黄昏
继续沉醉
狂人山庄
漆黑的空间
城市猎人
午夜流浪
过去与今天
缺口
仍然是要闯
坚持信念
千金一刻
谁来主宰
怀念您
完全地爱吧
可知道
我早应该习惯
金属狂人
你知道我的迷惘 (真的爱你)
永远等待
We Don't Wanna Make It Without You
不见不散
相依的心
温暖的家乡
Bye Bye
厌倦寂寞
快乐王国
请将手放开
醒你
心内心外
伤口
勇闯新世界
Love
明日の约束
最后的对话
忘记你
Myth
谁命我名字
祝您愉快
妄想
リゾ·ラバ ～International～
现代舞台
灰色的心
活得精彩
时日无多
怀念你
声音
关心永远在
爱妳一切
我的知己在街头
谁是勇敢
明日世界
门外看
候诊室
Cryin'
东方宝藏


# main

In [58]:
singer_list = [
        'mayday', 'jaychou', 'liyuchun', 'chenyixun', 'renxianqi', 'linjunjie',
        'sunyanzi', 'remen', 'fangwenshan', 'chenxinhong', 'caiyilin', 'wubai', 'zhoushen', 'zhoushen_pure', 'fenghuangchuanqi', 'wanglihong', 'liangjingru', 'wangxinling', 'twins', 'beyond', 'wuyuetian', 'luodayou', 'fangdatong'
    ]
file_path_prefix = f"data/{singer_list[-1]}/"
# file_path_prefix = f"data/renxianqi/"

In [59]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,5293866,003cSLOO35W3yP,特别的人,NaN,方大同,1066,003zHcYF44FVEV,危险世界,456882,000GvdTR0qcKh8,259,1397145600,特别的人,特别的人,危险世界,2014-04-11,2014
1,415824,0028XLMG07oUms,Love Song,NaN,方大同,1066,003zHcYF44FVEV,未来,33619,003Ctb8A447AgL,269,1198771200,LoveSong,lovesong,未来,2007-12-28,2007
2,510240500,003BBkww3jWdeI,才二十三,NaN,方大同,1066,003zHcYF44FVEV,梦想家 The Dreamer,56828085,002abjXr4Gc7sw,224,1724947200,才二十三,才二十三,梦想家 The Dreamer,2024-08-30,2024
3,2550926,001lmD2a2YZYN3,麦恩莉,NaN,方大同,1066,003zHcYF44FVEV,回到未来,192334,000ih4iv3HCmno,288,1353600000,麦恩莉,麦恩莉,回到未来,2012-11-23,2012
4,186041,002IFYcy4UMZNg,爱爱爱,NaN,方大同,1066,003zHcYF44FVEV,爱爱爱,15997,002kkyPO1moicK,213,1167321600,爱爱爱,爱爱爱,爱爱爱,2006-12-29,2006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,186053,002HoYFU1CJPfo,Love Outrolude (纯音乐),NaN,方大同,1066,003zHcYF44FVEV,爱爱爱,15997,002kkyPO1moicK,92,1167321600,LoveOutrolude,loveoutrolude,爱爱爱,2006-12-29,2006
125,149094,00168srj3jDbAv,跳,NaN,方大同,1066,003zHcYF44FVEV,Soulboy,13207,004J3oTJ2E4cuo,213,1132070400,跳,跳,Soulboy,2005-11-16,2005
126,108282169,002Wqlc83v0EbV,Once,NaN,方大同,1066,003zHcYF44FVEV,JTW 西游记 (Gold) [Explicit],1582772,000BjEpu1wy8th,171,1638892800,Once,once,JTW 西游记 (Gold) [Explicit],2021-12-08,2021
127,829563,004Iqqlc411g7A,Over Reprise,NaN,方大同,1066,003zHcYF44FVEV,15,70039,000HVhyy49acT8,167,1303228800,OverReprise,overreprise,15,2011-04-20,2011


In [ ]:
# 五月天需要使用word_to_fix
# if file_path_prefix == "data/mayday/":
#     df_word = lyric_words_process(file_path_prefix, word_to_fix)
# else:
#     df_word = lyric_words_process(file_path_prefix, word_to_fix=None)

In [12]:
# 词性解析
# hanlp暂时不需要word_to_fix
df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
df_word.to_csv(file_path_prefix + "raw_words_data.csv", index=False)
df_word

后来的我们
步步
突然好想你
知足
倔强
如果我们不曾相遇
温柔
你不是真正的快乐
任性
派对动物
恋爱ing
玫瑰少年
伤心的人别听慢歌 (贯彻快乐)
仓颉
星空
OAOA (现在就是永远)
任意门
离开地球表面
好好 (想把你写成一首歌)
拥抱
盛夏光年
转眼
因为你 所以我
顽固
如烟
刻在我心底的名字
一颗苹果
让我照顾你
为你写下这首情歌
勇敢
笑忘歌
听不到
成名在望
天使
终于结束的起点
你是唯一 (2013新录制作品)
第二人生
时光机
志明与春娇
候鸟
入阵曲
孙悟空
忘词
超人
将军令
我心中尚未崩坏的地方
宠上天
DNA
约翰蓝侬
夜访吸血鬼
人生海海
最重要的小事
憨人
我不愿 让你一个人
最好的一天
而我知道
诺亚方舟
咸鱼
终结孤单
我又初恋了
后青春期的诗
少年他的奇幻漂流
香水
洋葱 (2013新录制作品)
小太阳
什么歌
有些事现在不做 一辈子都不会做了
回来吧
爱情万岁
相信
洗衣机
米老鼠
疯狂世界
恒星的恒心
九号球
噢买尬
生命有一种绝对
干杯
爱情的模样
错错错
纯真
开天窗
我们
年轻就要对味
出头天
牙关
三个傻瓜
生存以上生活以下
为爱而生
一千个世纪
透露
2012
雌雄同体
乱世浮生
轧车
彩虹
放肆
垃圾车
好不好
心中无别人
兄弟
人生有限公司
明白
歪腰
I Love You 无望
罗密欧与茱丽叶
満ち足りた想い出
雨眠
花
有你的将来
麦来乱
T1 21 31 21
由我们主宰
永远的永远
摩托车日记
春天的呐喊
小时候
别惹我
君は幸せじゃないのに
爆肝
圣诞夜惊魂
不见不散
借问众神明
嘿！我要走了
反而
闯
能不能不要说
叫我第一名
阿姆斯壮


,song_id,word,pos,freq
0,107709592,的,u,20
1,107709592,了,y,15
2,107709592,后来,t,15
3,107709592,着,u,14
4,107709592,你,r,10
...,...,...,...,...
13921,4932219,没有,v,1
13922,4932219,关系,n,1
13923,4932219,可以,v,1
13924,4932219,接受,v,1


In [60]:
# 重新读取
df_word_read = pd.read_csv(file_path_prefix + "raw_words_data.csv")
df_word_read

,song_id,word,pos,freq
0,5293866,的,u,25
1,5293866,有,v,10
2,5293866,要,v,9
3,5293866,一,m,7
4,5293866,你,r,7
...,...,...,...,...
13827,351781404,Tony,nx,1
13828,351781404,Parker,nx,1
13829,351781404,on,nx,1
13830,351781404,the,nx,1


In [61]:
df_merged = words_data_merge(df_word_read, df_songs)
df_merged = df_merged.dropna(subset='song_name')
df_merged

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,5293866,的,u,25,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,危险世界,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0
1,5293866,有,v,10,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,危险世界,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0
2,5293866,要,v,9,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,危险世界,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0
3,5293866,一,m,7,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,危险世界,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0
4,5293866,你,r,7,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,危险世界,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13827,351781404,Tony,nx,1,0,003pAwqf1cVLDU,清楚点,NaN,方大同,1066.0,...,清楚点,26873253.0,002ZXgUe48Upou,214.0,1.649866e+09,清楚点,清楚点,清楚点,2022-04-14,2022.0
13828,351781404,Parker,nx,1,0,003pAwqf1cVLDU,清楚点,NaN,方大同,1066.0,...,清楚点,26873253.0,002ZXgUe48Upou,214.0,1.649866e+09,清楚点,清楚点,清楚点,2022-04-14,2022.0
13829,351781404,on,nx,1,0,003pAwqf1cVLDU,清楚点,NaN,方大同,1066.0,...,清楚点,26873253.0,002ZXgUe48Upou,214.0,1.649866e+09,清楚点,清楚点,清楚点,2022-04-14,2022.0
13830,351781404,the,nx,1,0,003pAwqf1cVLDU,清楚点,NaN,方大同,1066.0,...,清楚点,26873253.0,002ZXgUe48Upou,214.0,1.649866e+09,清楚点,清楚点,清楚点,2022-04-14,2022.0


In [62]:
# 过滤中文词
df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
df_merged_chn

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,5293866,的,u,25,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,危险世界,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0
1,5293866,有,v,10,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,危险世界,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0
2,5293866,要,v,9,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,危险世界,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0
3,5293866,一,m,7,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,危险世界,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0
4,5293866,你,r,7,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,危险世界,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13822,351781404,总会,v,1,1,003pAwqf1cVLDU,清楚点,NaN,方大同,1066.0,...,清楚点,26873253.0,002ZXgUe48Upou,214.0,1.649866e+09,清楚点,清楚点,清楚点,2022-04-14,2022.0
13823,351781404,深浅,n,1,1,003pAwqf1cVLDU,清楚点,NaN,方大同,1066.0,...,清楚点,26873253.0,002ZXgUe48Upou,214.0,1.649866e+09,清楚点,清楚点,清楚点,2022-04-14,2022.0
13824,351781404,真是,v,1,1,003pAwqf1cVLDU,清楚点,NaN,方大同,1066.0,...,清楚点,26873253.0,002ZXgUe48Upou,214.0,1.649866e+09,清楚点,清楚点,清楚点,2022-04-14,2022.0
13825,351781404,假,a,1,1,003pAwqf1cVLDU,清楚点,NaN,方大同,1066.0,...,清楚点,26873253.0,002ZXgUe48Upou,214.0,1.649866e+09,清楚点,清楚点,清楚点,2022-04-14,2022.0


In [63]:
# 查看歌曲数
df_merged_chn['song_name_pure'].nunique()

112

In [64]:
# 虚拟专辑数据
df_songs_part = df_merged_chn[[
    'song_name_pure'
]].drop_duplicates(keep='first').reset_index(drop=True)
# 只保留120个
df_songs_part = df_songs_part.head(100)
df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                         1).astype(str)
df_songs_part['album_order'] = df_songs_part.index // 10
df_songs_part

,song_name_pure,album_name,album_order
0,特别的人,PART 1,0
1,lovesong,PART 1,0
2,才二十三,PART 1,0
3,麦恩莉,PART 1,0
4,爱爱爱,PART 1,0
...,...,...,...
95,赶场,PART 10,9
96,妈妈说,PART 10,9
97,认识你,PART 10,9
98,gottamakeachange,PART 10,9


In [65]:
# 虚拟专辑数据，index//12+1作为虚拟专辑
df_merged_chn = df_merged_chn.copy()
df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
df_merged_chn = df_merged_chn.drop(columns=['album_name'])
df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
# 删除album_order为空的数据
df_merged_chn = df_merged_chn.dropna(subset=['album_order'], axis=0)
df_merged_chn

,song_id,word,pos,freq,is_chinese,song_mid,song_name,song_subname,artist_name,artist_id,...,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year,album_name_raw,album_name,album_order
0,5293866,的,u,25,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0,危险世界,PART 1,0.0
1,5293866,有,v,10,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0,危险世界,PART 1,0.0
2,5293866,要,v,9,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0,危险世界,PART 1,0.0
3,5293866,一,m,7,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0,危险世界,PART 1,0.0
4,5293866,你,r,7,1,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,...,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0,危险世界,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9500,5302749,真伪,n,1,1,002SZPRw2CzomW,僵尸,NaN,方大同,1066.0,...,290.0,1.397146e+09,僵尸,僵尸,危险世界,2014-04-11,2014.0,危险世界,PART 10,9.0
9501,5302749,天花乱坠,i,1,1,002SZPRw2CzomW,僵尸,NaN,方大同,1066.0,...,290.0,1.397146e+09,僵尸,僵尸,危险世界,2014-04-11,2014.0,危险世界,PART 10,9.0
9502,5302749,你,r,1,1,002SZPRw2CzomW,僵尸,NaN,方大同,1066.0,...,290.0,1.397146e+09,僵尸,僵尸,危险世界,2014-04-11,2014.0,危险世界,PART 10,9.0
9503,5302749,该,v,1,1,002SZPRw2CzomW,僵尸,NaN,方大同,1066.0,...,290.0,1.397146e+09,僵尸,僵尸,危险世界,2014-04-11,2014.0,危险世界,PART 10,9.0


In [66]:
df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

In [67]:
# 数据查验
songs_n = df_merged_chn[df_merged_chn['pos'] == 'n']['song_name_pure'].unique().tolist()
songs_all = df_merged_chn['song_name_pure'].unique().tolist()
for i in songs_all:
    if i not in songs_n:
        print(i)

# 歌曲数据更新

In [68]:
df_songs_final = df_merged_chn.drop(columns=['word', 'pos', 'freq', 'is_chinese']).drop_duplicates().reset_index(drop=True)

df_songs_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year,album_name_raw,album_name,album_order
0,5293866,003cSLOO35W3yP,特别的人,NaN,方大同,1066.0,003zHcYF44FVEV,456882.0,000GvdTR0qcKh8,259.0,1.397146e+09,特别的人,特别的人,危险世界,2014-04-11,2014.0,危险世界,PART 1,0.0
1,415824,0028XLMG07oUms,Love Song,NaN,方大同,1066.0,003zHcYF44FVEV,33619.0,003Ctb8A447AgL,269.0,1.198771e+09,LoveSong,lovesong,未来,2007-12-28,2007.0,未来,PART 1,0.0
2,510240500,003BBkww3jWdeI,才二十三,NaN,方大同,1066.0,003zHcYF44FVEV,56828085.0,002abjXr4Gc7sw,224.0,1.724947e+09,才二十三,才二十三,梦想家 The Dreamer,2024-08-30,2024.0,梦想家 The Dreamer,PART 1,0.0
3,2550926,001lmD2a2YZYN3,麦恩莉,NaN,方大同,1066.0,003zHcYF44FVEV,192334.0,000ih4iv3HCmno,288.0,1.353600e+09,麦恩莉,麦恩莉,回到未来,2012-11-23,2012.0,回到未来,PART 1,0.0
4,186041,002IFYcy4UMZNg,爱爱爱,NaN,方大同,1066.0,003zHcYF44FVEV,15997.0,002kkyPO1moicK,213.0,1.167322e+09,爱爱爱,爱爱爱,爱爱爱,2006-12-29,2006.0,爱爱爱,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,149096,00332QZE04gpjh,赶场,NaN,方大同,1066.0,003zHcYF44FVEV,13207.0,004J3oTJ2E4cuo,213.0,1.132070e+09,赶场,赶场,Soulboy,2005-11-16,2005.0,Soulboy,PART 10,9.0
96,2550929,000GD6933sXD7h,妈妈说,NaN,方大同,1066.0,003zHcYF44FVEV,192334.0,000ih4iv3HCmno,247.0,1.353600e+09,妈妈说,妈妈说,回到未来,2012-11-23,2012.0,回到未来,PART 10,9.0
97,931490,003ynVVU48vOpD,认识你,Hidden Track,方大同,1066.0,003zHcYF44FVEV,13207.0,004J3oTJ2E4cuo,331.0,1.132070e+09,认识你,认识你,Soulboy,2005-11-16,2005.0,Soulboy,PART 10,9.0
98,106725491,002bz5Xf1lZ7kW,Gotta Make A Change,NaN,方大同,1066.0,003zHcYF44FVEV,70039.0,000HVhyy49acT8,295.0,1.303229e+09,GottaMakeAChange,gottamakeachange,15,2011-04-20,2011.0,15,PART 10,9.0


In [69]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# 测试

In [25]:
1260/500

2.52

In [26]:
2318/2.52

919.8412698412699

In [57]:
from datetime import datetime
datetime.now().month

3